# Working with Text Data

An LLM cannot read raw text. Before training, the text goes through a pipeline:

1. **Tokenization**: split the text into small units (tokens).
2. **Token IDs**: map each token to an integer using a vocabulary.
3. **Embeddings**: turn each ID into a vector the model can learn from.

This notebook covers steps 1 and 2 with a simple tokenizer built by hand.

![Diagram](assets/llm-text-pipeline.svg)

## Tokenizing text

A token is a piece of text: a word, a punctuation mark, or part of a word.

To see tokenization in practice I use https://tiktokenizer.vercel.app/?model=gpt2. It shows how a sentence is split and which token ID each piece receives.

![Diagram](assets/tokenizer-steps.svg)

### Loading the dataset

The text used here is *The Verdict*, a short story by Edith Wharton in the public domain. It is small enough to run everything on a laptop. The download only happens if the file is not already on disk.

In [122]:
import os
import urllib.request

file_path = "the-verdict.txt"

if not os.path.exists(file_path):
    url = (
        "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/"
        "refs/heads/main/ch02/01_main-chapter-code/the-verdict.txt"
    )
    urllib.request.urlretrieve(url, file_path)

In [123]:
with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

In [124]:
print(raw_text[:99])

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [125]:
print("Total number of characters:", len(raw_text))

Total number of characters: 20479


### Splitting on whitespace

First attempt: `re.split` on whitespace. The capture group `(\s)` keeps the separators in the result. The problem is that punctuation stays glued to the words (`Hello,` and `world.`).

In [126]:
import re

text = "Hello, world. My first test in tokenization"
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world.', ' ', 'My', ' ', 'first', ' ', 'test', ' ', 'in', ' ', 'tokenization']


### Splitting on punctuation

Adding `,` and `.` to the pattern separates punctuation into its own tokens. The side effect is a few empty strings in the list.

In [127]:
result = re.split(r'([,.]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'My', ' ', 'first', ' ', 'test', ' ', 'in', ' ', 'tokenization']


### Removing whitespace

Empty strings and whitespace carry no information for this tokenizer, so they are dropped. This is a trade-off: it saves memory and compute, but text where exact spacing matters (Python code, for example) would need the whitespace kept as tokens.

In [128]:
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'My', 'first', 'test', 'in', 'tokenization']


### Handling more punctuation

The final pattern also covers question marks, quotes, parentheses and the double dash `--` that appears in the story.

In [129]:
text = "Hello, world. Is this -- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


### Tokenizing the whole story

Now the same pattern is applied to `raw_text` instead of the sample sentence. The result, `preprocessed`, is the list of tokens used in the rest of the chapter.

In [130]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

print(len(preprocessed))
print(preprocessed[:30])

4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## Converting tokens into token IDs

The model works with numbers, not strings. The vocabulary is the set of unique tokens, sorted, where each one gets an integer ID.

![Diagram](assets/vocabulary-creation.svg)

In [131]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

1130


In [132]:
vocab = {token:integer for integer,token in enumerate(all_words)}
vocab

{'!': 0,
 '"': 1,
 "'": 2,
 '(': 3,
 ')': 4,
 ',': 5,
 '--': 6,
 '.': 7,
 ':': 8,
 ';': 9,
 '?': 10,
 'A': 11,
 'Ah': 12,
 'Among': 13,
 'And': 14,
 'Are': 15,
 'Arrt': 16,
 'As': 17,
 'At': 18,
 'Be': 19,
 'Begin': 20,
 'Burlington': 21,
 'But': 22,
 'By': 23,
 'Carlo': 24,
 'Chicago': 25,
 'Claude': 26,
 'Come': 27,
 'Croft': 28,
 'Destroyed': 29,
 'Devonshire': 30,
 'Don': 31,
 'Dubarry': 32,
 'Emperors': 33,
 'Florence': 34,
 'For': 35,
 'Gallery': 36,
 'Gideon': 37,
 'Gisburn': 38,
 'Gisburns': 39,
 'Grafton': 40,
 'Greek': 41,
 'Grindle': 42,
 'Grindles': 43,
 'HAD': 44,
 'Had': 45,
 'Hang': 46,
 'Has': 47,
 'He': 48,
 'Her': 49,
 'Hermia': 50,
 'His': 51,
 'How': 52,
 'I': 53,
 'If': 54,
 'In': 55,
 'It': 56,
 'Jack': 57,
 'Jove': 58,
 'Just': 59,
 'Lord': 60,
 'Made': 61,
 'Miss': 62,
 'Money': 63,
 'Monte': 64,
 'Moon-dancers': 65,
 'Mr': 66,
 'Mrs': 67,
 'My': 68,
 'Never': 69,
 'No': 70,
 'Now': 71,
 'Nutley': 72,
 'Of': 73,
 'Oh': 74,
 'On': 75,
 'Once': 76,
 'Only': 77,
 '

In [133]:
vocab["sunburn"]

954

In [134]:
into_to_str = {i:s for s,i in vocab.items()}
into_to_str[954]

'sunburn'

### A simple tokenizer class

`encode` splits the text into tokens and maps each one to its ID. `decode` does the reverse: IDs back to tokens, joined with spaces, then the space before punctuation is removed.

![Diagram](assets/tokenizer-encode-decode.svg)

In [135]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [136]:
tokenizer = SimpleTokenizerV1(vocab)
text = """It's The last the painted, you know,"
          Mrs. Gisburn said with pardonable pride."""

In [137]:
ids = tokenizer.encode(text)
print(ids)

[56, 2, 850, 93, 602, 988, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [138]:
tokenizer.decode(ids)

'It\' s The last the painted, you know," Mrs. Gisburn said with pardonable pride.'

## Adding special context tokens

`SimpleTokenizerV1` only knows the tokens found in *The Verdict*. Any word outside the vocabulary, like `Hello`, makes `encode` fail with a `KeyError`. Special tokens solve this and also give the model extra context about the text.

Two tokens are added to the vocabulary:

- `<|unk|>`: replaces any word that is not in the vocabulary.
- `<|endoftext|>`: marks the boundary between two unrelated texts. When several documents are concatenated for training, it tells the model that what comes next is not a continuation of what came before.

Both go at the end of the sorted vocabulary, so the vocabulary grows from 1130 to 1132 entries.

How `encode` handles each token once `<|unk|>` exists:

![Unknown-token handling](assets/unknown-token-handling.svg)

How independent texts are joined for training:

![End-of-text boundaries](assets/end-of-text-boundaries.svg)

Other tokens that show up in LLMs:

- `[BOS]` (beginning of sequence): marks where a text starts.
- `[EOS]` (end of sequence): marks where a text ends, same role as `<|endoftext|>`.
- `[PAD]` (padding): fills shorter texts so every text in a batch has the same length.

GPT models keep it simple and use only `<|endoftext|>`, including for padding. They also have no `<|unk|>` token: instead of replacing unknown words, they use byte pair encoding (BPE), which breaks any word into smaller known pieces.

In [139]:
text = "Hello, do you like tea. is this -- a test?"
tokenizer.encode(text)

#This problem below is normal, because the word Hello haven't in variable

KeyError: 'Hello'

In [140]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<endoftext|>", "<|unk|>"])

vocab = {token:interger for interger,token in enumerate(all_tokens)}

In [141]:
len(vocab.items())

1132

In [142]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<endoftext|>', 1130)
('<|unk|>', 1131)


In [143]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]

        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [144]:
tokenizer = SimpleTokenizerV2(vocab)

In [145]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 7, 584, 999, 6, 115, 1131, 10]

In [146]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea. is this -- a <|unk|>?'

## Byte Pair Encoding (BPE)

BPE turns text into tokens without needing to know every full word.

It starts with small units and joins the pairs that appear most often in the text. Frequent parts such as `tion`, `ment`, or `ing` can then become a single token.

![How BPE creates tokens](assets/bpe-merges.svg)

When BPE reads new text, it looks for the largest pieces it already knows. If a complete word is not in the vocabulary, it can still split the word into smaller pieces.

![How BPE tokenizes a new word](assets/bpe-unknown-word.svg)

Because bytes are always available, any text can be converted into tokens. This is why GPT tokenizers usually do not need an unknown-word token.


In [147]:
import tiktoken

In [148]:
#!uv pip install tiktoken==0.9.0

In [149]:
tiktoken.__version__

'0.14.0'

In [150]:
tokenizer = tiktoken.get_encoding("gpt2")

In [151]:
tokenizer.encode("Hello world")

[15496, 995]

In [152]:
tokenizer.decode(tokenizer.encode("Hello world"))

'Hello world'

In [153]:
text = (
    "Hello, do you like? <|endoftext|> In the sunlit terraces"
    "Of someunknowPlace."
    )
tokenizer.encode(text, allowed_special={"<|endoftext|>"})

[15496,
 11,
 466,
 345,
 588,
 30,
 220,
 50256,
 554,
 262,
 4252,
 18250,
 8812,
 2114,
 5189,
 617,
 2954,
 2197,
 27271,
 13]

## Data sampling with a sliding window

An LLM is trained on a single task: predict the next token. To train it, the encoded text has to be turned into input-target pairs, where the target is the input shifted one position to the right.

Using the words of "In the sunlit terraces", at each step the model sees one more token and learns to predict the one that follows:

```
In                    --->  the
In the                --->  sunlit
In the sunlit         --->  terraces
```

In practice the pairs are made of token IDs, not words, and they come from a window that slides over the encoded text. Two parameters control it:

- `context_size` (`max_length`): how many tokens go into each input. It defines how much text the model sees at once.
- `stride`: how many positions the window moves between one sample and the next. With `stride = context_size` the windows do not overlap. A smaller stride creates overlap: more samples from the same text, but more repeated data and a higher risk of overfitting.

With `context_size = 4` and `stride = 4`:

```
tokens:  t1 t2 t3 t4 t5 t6 t7 t8 t9

x = [t1 t2 t3 t4]    y = [t2 t3 t4 t5]
x = [t5 t6 t7 t8]    y = [t6 t7 t8 t9]
```

The target is the whole shifted window, not a single token, because each input of length `context_size` holds `context_size` prediction tasks at once: position `i` of `y` is the next token after `x[:i+1]`.

The steps:

1. Encode the whole story with the GPT-2 BPE tokenizer.
2. Take a window of `context_size` tokens as the input `x`.
3. Take the same window shifted by one token as the target `y`.
4. Move the window by `stride` and repeat until the end of the text.

In [154]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [155]:
enc_sample = enc_text[50:]

In [156]:
len(enc_sample)

5095

In [157]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y: {y}")

x: [290, 4920, 2241, 287]
y: [4920, 2241, 287, 257]


In [158]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->",tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [159]:
import torch

In [160]:
torch.__version__

'2.14.0+cu130'

In [161]:
pip install torch==2.6.0

/home/aurelio.malheiros/projects/personal/llm/build-llm/.venv/bin/python3: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [162]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


In [163]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")

    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [164]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [165]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [166]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [168]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
